In [ ]:
from getpass import getpass

admin_rdm_url = 'https://admin.staging.rdm.example.com/'

idp_name_integrated_admin = None
idp_username_integrated_admin_2 = None
idp_password_integrated_admin_2 = None
integrated_admin_user_name = 'admin@example.com'

default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_name_integrated_admin is None:
    idp_name_integrated_admin = input(prompt=f'IdP Name for integrated admin: ')
if idp_username_integrated_admin_2 is None:
    idp_username_integrated_admin_2 = input(prompt=f'Username for {idp_name_integrated_admin} (統合管理者2)')
if idp_password_integrated_admin_2 is None:
    idp_password_integrated_admin_2 = getpass(prompt=f'Password for {idp_username_integrated_admin_2}@{idp_name_integrated_admin} (統合管理者2)')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# クリーンアップ処理

- サブシステム名: 管理者
- ページ/アドオン: 管理者
- 機能分類: 認証/認可
- シナリオ名: クリーンアップ処理
- 用意するテストデータ:   
   ・GRDM 管理者のURL   
   ・アカウント(統合管理者2）    
   ・アカウント(統合管理者）のユーザー名    
   ・USE_LOGIN_FORM が空（ENABLE_LOGIN_FORM が偽）の環境

In [ ]:
import importlib

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGakuNin RDM管理者のトップページを表示する

・`/account/login/` にリダイレクトされ、ログイン画面が表示されること   
・リダイレクト後のURLが「https://{admin_domain}/account/login/」であること

In [ ]:
import re
from urllib.parse import urlparse

async def _step(page):
    await page.goto(admin_rdm_url)

    await expect(page.locator('.login-logo')).to_be_visible(timeout=transition_timeout)

    login_url = page.url
    parsed_url = urlparse(login_url)
    assert re.fullmatch(
        r"/account/login/",
        parsed_url.path,
    ), f"リダイレクト後のURLが期待する形式ではありません。actual: {login_url}"

await run_pw(_step)

## アカウント(統合管理者2)を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。   

In [ ]:
async def _step(page):
    await scripts.grdm.login_as_admin(
        page, idp_name_integrated_admin, idp_username_integrated_admin_2, idp_password_integrated_admin_2, transition_timeout=transition_timeout
    )

    await expect(page.locator('//*[contains(@class, "btn-danger") and contains(text(), "ログアウト")]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## ブラウザのアドレスバーに、「https://{admin_domain}/admin/osf/osfuser」を直接入力する	

「変更する osf user を選択」ページが表示されること   
※ 画面見出し: 変更する osf user を選択

In [ ]:
async def _step(page):
    await page.goto(f"{admin_rdm_url.rstrip('/')}/admin/osf/osfuser")

    await expect(page.get_by_role("heading", name="変更する osf user を選択")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「変更する osf user を選択」ページに、アカウント(統合管理者)のユーザー名をクリックする

「osf user を変更」ページが表示されること   
「Is registered」と「Is active」のチェックボックスがチェックされていないこと   
※ 画面見出し: osf user を変更

In [ ]:
async def _step(page):
    user_link = page.get_by_role("link", name=integrated_admin_user_name, exact=True)
    if await user_link.count() == 0:
        await page.get_by_role("link", name="全件表示", exact=True).click()
        await expect(user_link).to_be_visible(timeout=transition_timeout)

    await user_link.click()

    await expect(page.get_by_role("heading", name="osf user を変更", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#id_is_registered")).not_to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_registered").scroll_into_view_if_needed()
    await expect(page.locator("#id_is_active")).not_to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_active").scroll_into_view_if_needed()

await run_pw(_step)

## 「osf user を変更」ページで、「Is registered」にチェックを入れ、「保存」ボタンをクリックする

「変更する osf user を選択」ページが表示されること   
「Osf user {アカウント(統合管理者)のユーザー名} を変更しました。」というメッセージが表示されること   
※ 画面見出し: 変更する osf user を選択   
※「保存」ボタンをクリックすると、「Is active」のチェック状態が「Is registered」と同じになること。「Is registered」がチェックされている場合、「Is active」もチェックされている状態になること。   
※ 実施による影響：【ソース上】Django 管理サイトのフォームで統合管理者のレコード全体を保存し直すため、日時の項目（date_confirmed、date_last_login 等）が秒未満を切り捨てた値で保存されます（Django 1.11 の SplitDateTimeWidget.decompress が microsecond=0 にするため）。date_confirmed は検索インデックスの更新対象（osf/models/user.py の SEARCH_UPDATE_FIELDS）のため、保存前の値に秒未満が含まれていた場合は、統合管理者と、統合管理者が参加しているプロジェクトの検索インデックスが更新されます。

In [ ]:
async def _step(page):
    await page.locator("#id_is_registered").check()
    await page.get_by_role("button", name="保存", exact=True).click()

    await expect(page.locator(".text-success")).to_contain_text(f'Osf user "{integrated_admin_user_name}" を変更しました。',timeout=transition_timeout)

await run_pw(_step)

## 「変更する osf user を選択」ページに、アカウント(統合管理者)のユーザー名をクリックする

「osf user を変更」ページが表示されること     
「Is registered」と「Is active」のチェックボックスがチェックされていること     
※ 画面見出し: osf user を変更   

In [ ]:

async def _step(page):
    await page.reload()
    user_link = page.get_by_role("link", name=integrated_admin_user_name, exact=True)
    if await user_link.count() == 0:
        await page.get_by_role("link", name="全件表示", exact=True).click()
        await expect(user_link).to_be_visible(timeout=transition_timeout)

    await user_link.click()

    await expect(page.get_by_role("heading", name="osf user を変更", exact=True)).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#id_is_registered")).to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_registered").scroll_into_view_if_needed()
    await expect(page.locator("#id_is_active")).to_be_checked(timeout=transition_timeout)
    await page.locator("#id_is_active").scroll_into_view_if_needed()

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}